In [0]:
%sql
create schema if not exists weather.silver;

In [0]:
source_table= "weather.bronze.forecast_weather"
target_table= "weather.silver.forecast_weather"

In [0]:
from pyspark.sql.types import *

forecast_schema = MapType(
    StringType(),
    StructType([
        StructField("air_quality", StructType([
            StructField("co", StringType(), True),
            StructField("gb-defra-index", StringType(), True),
            StructField("no2", StringType(), True),
            StructField("o3", StringType(), True),
            StructField("pm10", StringType(), True),
            StructField("pm2_5", StringType(), True),
            StructField("so2", StringType(), True),
            StructField("us-epa-index", StringType(), True)
        ]), True),

        StructField("astro", StructType([
            StructField("moon_illumination", LongType(), True),
            StructField("moon_phase", StringType(), True),
            StructField("moonrise", StringType(), True),
            StructField("moonset", StringType(), True),
            StructField("sunrise", StringType(), True),
            StructField("sunset", StringType(), True)
        ]), True),

        StructField("avgtemp", LongType(), True),
        StructField("date", StringType(), True),
        StructField("date_epoch", LongType(), True),
        StructField("maxtemp", LongType(), True),
        StructField("mintemp", LongType(), True),
        StructField("sunhour", LongType(), True),
        StructField("totalsnow", LongType(), True),
        StructField("uv_index", LongType(), True)
    ])
)
StructField("forecast", forecast_schema, True)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
df = spark.read.table(source_table)

forecast_df = df.withColumn(
    "forecast_map",
    from_json(to_json(col("data.forecast")), MapType(StringType(), forecast_schema.valueType))
).select(
    col("city"),
    explode(col("forecast_map")).alias("forecast_date", "forecast")
)
display(forecast_df)

In [0]:

from pyspark.sql.functions import col, explode,round

df = spark.read.table(source_table)

# The forecast field is a struct with date-named fields
# Convert to JSON, then parse as map to enable explode
from pyspark.sql.functions import from_json, to_json
from pyspark.sql.types import MapType, StringType

forecast_df = df.withColumn(
    "forecast_map",
    from_json(to_json(col("data.forecast")), MapType(StringType(), forecast_schema.valueType))
).select(
    col("city"),
    explode(col("forecast_map")).alias("forecast_date", "forecast")
).select(
    col("city"),
    col("forecast_date"),
    col("forecast.avgtemp"),
    col("forecast.maxtemp"),
    col("forecast.mintemp"),
    col("forecast.sunhour"),
    col("forecast.uv_index"),
    round(col("forecast.air_quality.co").cast("double"),2).alias("co"),
    round(col("forecast.air_quality.pm10").cast("double"),2).alias("pm10")
)
display(forecast_df)


In [0]:
forecast_df.write.format("delta").mode("append").saveAsTable(target_table)

In [0]:
%sql
select * from weather.silver.forecast_weather